# Chroma RAG Training Pipeline

This notebook provides the full 4-step pipeline for Chroma:

1. **Point to dataset** - Chunk documents and upload to a Chroma server with vector embeddings
2. **Create synthetic QA** - Generate question-answer pairs with CgftPipeline
3. **Create env** - Configure the Chroma search environment
4. **Run training job** - Launch the training with remote validation

Chroma auto-embeds with its built-in model (`all-MiniLM-L6-v2`) by default. You can optionally provide a custom embedding function (e.g. Azure OpenAI).

**Requires a running Chroma server** — see the `chroma_local_vs_remote.ipynb` notebook for local/in-memory testing.

## Setup

Install dependencies and configure API credentials.

In [1]:
# For Google Colab - clone repo and install
# !git clone https://github.com/cgftinc/synthetic-data-prep-lib.git
# %cd synthetic-data-prep-lib
# %pip install -e ".[chroma]" openai

In [2]:
%load_ext autoreload
%autoreload 2

import sys
print(f"Python: {sys.executable}")

Python: /Users/thariqridha/.venvs/cgft-chroma/bin/python


In [3]:
import cgft
from cgft.utils import ensure_safe_python_version

ensure_safe_python_version()
print(f"cgft: {cgft.__file__}")

cgft: /Users/thariqridha/Projects/cgft_projects/cgft/.claude/worktrees/chroma/src/cgft/__init__.py


In [4]:
# Configuration

# Chroma server — must be running and accessible.
# Start locally with: chroma run --host localhost --port 8000 --path /tmp/chroma_data
CHROMA_HOST = "localhost"
CHROMA_PORT = 8000
COLLECTION_NAME = "cgft-rag-test"

# CGFT API key — create one at https://app.cgft.io/account/api-keys
API_KEY = ""
BASE_URL = "https://app.cgft.io"

# LLM API credentials — used by CgftPipeline for QA generation, filtering, and refinement
LLM_API_KEY = ""
LLM_BASE_URL = "https://judge-elsa.services.ai.azure.com/openai/v1/"
LLM_MODEL = "grok-4-fast-reasoning"

# Judge LLM credentials — used at reward time to evaluate correctness
JUDGE_API_KEY = LLM_API_KEY
JUDGE_BASE_URL = LLM_BASE_URL
JUDGE_MODEL = "grok-4-fast-reasoning"

# Dataset configuration
# Point to a local directory containing markdown documents.
# See docs.cgft.io/rag/synthetic_datagen for sample documents.
DOCS_PATH = "./samples/posthog/docs/"

# Corpus intent used for corpus profiling (summary + example queries)
CORPUS_DESCRIPTION = "Posthog documentation including docs, company policy, etc."
EXAMPLE_QUERIES = [
    "how to feature flag",
    "setup reverse proxy to avoid ad blockers",
    "posthog install nextjs",
]

# QA generation configuration
TOTAL_SAMPLES = 30
OUTPUT_DIR = "outputs/chroma_rag"

# Optional: name for your training experiment
EXPERIMENT_NAME = None  # e.g. "posthog-chroma-v1"
EXPERIMENT_PREFIX = None  # e.g. "posthog-chroma"

# Optional: custom embedding function (e.g. Azure OpenAI).
# Set to None to use Chroma's built-in all-MiniLM-L6-v2 embeddings.
EMBED_FN = None

# To use Azure OpenAI embeddings instead, uncomment and fill in:
# from openai import AzureOpenAI
# _openai = AzureOpenAI(api_key="...", api_version="2024-02-01", azure_endpoint="...")
# def EMBED_FN(texts):
#     response = _openai.embeddings.create(model="text-embedding-3-small", input=texts)
#     return [item.embedding for item in sorted(response.data, key=lambda x: x.index)]

## Step 0: Verify Chroma Server

Confirm the server is reachable and test the embedding setup.

In [6]:
import chromadb
print(f"chromadb version: {chromadb.__version__}")

# Test server connection
client = chromadb.HttpClient(host=CHROMA_HOST, port=CHROMA_PORT)
heartbeat = client.heartbeat()
print(f"Server heartbeat: {heartbeat}")
print(f"Connected to Chroma at {CHROMA_HOST}:{CHROMA_PORT}")

# Test embedding (either custom or Chroma's built-in)
if EMBED_FN is not None:
    test_vec = EMBED_FN(["hello world"])
    print(f"\nCustom embed_fn: dimension={len(test_vec[0])}")
else:
    print("\nUsing Chroma's built-in embeddings (all-MiniLM-L6-v2)")
    print("No external embedding credentials needed.")

chromadb version: 1.5.5
Server heartbeat: 1774396952883333000
Connected to Chroma at localhost:8000

Using Chroma's built-in embeddings (all-MiniLM-L6-v2)
No external embedding credentials needed.


## Step 1: Point to Dataset

Chunk your documents and upload to a Chroma collection.

> **Existing collection?** If you already have a populated collection, skip `populate_from_folder` and just create the `ChromaChunkSource` pointing to it.

In [7]:
from cgft.corpus.chroma.source import ChromaChunkSource

source = ChromaChunkSource(
    collection_name=COLLECTION_NAME,
    host=CHROMA_HOST,
    port=CHROMA_PORT,
    embed_fn=EMBED_FN,
    enable_bm25=True,  # auto-degrades if server doesn't support it
)

source.populate_from_folder(DOCS_PATH)

Chunking documents from ./samples/posthog/docs/...
ChunkCollection Summary
Total chunks: 3498
Total files: 1264

Chunk Size Statistics:
  Min: 29 chars (contribute/badge.md, chunk 0)
  Max: 2048 chars (data-warehouse/query.mdx, chunk 0)
  Mean: 1150 chars

File Structure:
├── data-warehouse/
│   ├── _snippets/
│   │   └── query-join-example.mdx (1 chunks)
│   ├── sources/
│   │   ├── reddit-ads.mdx (1 chunks)
│   │   ├── s3.mdx (1 chunks)
│   │   ├── temporal.mdx (1 chunks)
│   │   ├── klaviyo.mdx (1 chunks)
│   │       ... 36 more files
│   ├── views/
│   │   ├── index.mdx (2 chunks)
│   │   └── materialize.mdx (2 chunks)
│   ├── sql/
│   │   ├── variables.mdx (2 chunks)
│   │   ├── index.mdx (8 chunks)
│   │   ├── data-access.mdx (3 chunks)
│   │   └── useful-functions.mdx (7 chunks)
│   ├── under-the-hood.md (1 chunks)
│   ├── cutting-costs.mdx (3 chunks)
│   ├── join.mdx (2 chunks)
│   ├── run-sql-mcp.mdx (4 chunks)
│       ... 8 more files
├── advanced/
│   ├── proxy/
│   │   ├── 

Uploading batches:   0%|          | 0/35 [00:00<?, ?it/s]


Upload complete! 3498 chunks written to collection 'cgft-rag-test'.


### Step 1a: Verify the corpus

Quick sanity checks: capabilities, sample chunks, search, file-structure awareness.

In [8]:
from pprint import pprint

# --- Capabilities ---
caps = source.get_search_capabilities()
print("=== Search Capabilities ===")
pprint(caps)
print(f"\nBackend: {caps['backend']}")
print(f"Modes: {caps['modes']}")
assert "vector" in caps["modes"], "Expected vector mode"
print("Capabilities check PASSED")

# --- Sample chunks ---
print("\n=== Sample Chunks ===")
samples = source.sample_chunks(n=3, min_chars=100)
print(f"Sampled {len(samples)} chunks (min 100 chars)")
for i, chunk in enumerate(samples):
    print(f"\n--- Chunk {i+1} ({len(chunk.content)} chars) ---")
    print(f"  file_path: {chunk.get_metadata('file_path')}")
    print(f"  chunk_index: {chunk.get_metadata('chunk_index')}")
    print(f"  content: {chunk.content[:120]}...")
assert len(samples) == 3, f"Expected 3 samples, got {len(samples)}"
print("\nSample check PASSED")

=== Search Capabilities ===
{'backend': 'chroma',
 'constraints': {'max_top_k': 10000, 'vector_dimensions': None},
 'filter_ops': {'field': {'lte', 'eq', 'in', 'gte'},
                'logical': {'and', 'or', 'not'}},
 'graph_expansion': False,
 'modes': {'vector'},
 'ranking': {'cosine'}}

Backend: chroma
Modes: {'vector'}
Capabilities check PASSED

=== Sample Chunks ===
Sampled 3 chunks (min 100 chars)

--- Chunk 1 (1230 chars) ---
  file_path: cdp/fivetran-airbyte.md
  chunk_index: 2
  content: ## Supported destinations  
PostHog batch exports support all major data warehouses:  
- [BigQuery](/docs/cdp/batch-expo...

--- Chunk 2 (1363 chars) ---
  file_path: feature-flags/targeting-groups.md
  chunk_index: 1
  content: ## Targeting pages  
Another target you might want is pages. For example, to ensure components are consistent across a s...

--- Chunk 3 (1503 chars) ---
  file_path: libraries/woocommerce.md
  chunk_index: 0
  content: ---
title: How to set up WooCommerce analytics w

In [9]:
from cgft.corpus.search_schema.search_types import SearchSpec, FieldPredicate, AndPredicate

# --- Vector search ---
print("=== Vector Search ===")
results = source.search(SearchSpec(
    mode="vector",
    top_k=3,
    text_query="how to set up feature flags",
))
print(f"Found {len(results)} results for 'how to set up feature flags'")
for i, chunk in enumerate(results):
    print(f"  {i+1}. ({len(chunk.content)} chars) {chunk.content[:120]}...")
assert len(results) > 0
print("Vector search PASSED")

# --- search_text() ---
print("\n=== search_text() ===")
text_results = source.search_text("reverse proxy setup", top_k=3)
print(f"Found {len(text_results)} results")
assert len(text_results) > 0
print("search_text PASSED")

# --- search_content() (cloudpickle-safe) ---
print("\n=== search_content() ===")
content_results = source.search_content(SearchSpec(
    mode="vector", top_k=3, text_query="analytics setup",
))
print(f"Got {len(content_results)} content strings")
assert all(isinstance(c, str) for c in content_results)
print("search_content PASSED")

# --- Lexical / Hybrid (if server supports it) ---
if "lexical" in caps["modes"]:
    print("\n=== Lexical (BM25) Search ===")
    lex_results = source.search(SearchSpec(mode="lexical", top_k=3, text_query="pip install posthog"))
    print(f"Found {len(lex_results)} results")
    print("Lexical search PASSED")

    print("\n=== Hybrid (RRF) Search ===")
    hyb_results = source.search(SearchSpec(mode="hybrid", top_k=3, text_query="feature flag analytics"))
    print(f"Found {len(hyb_results)} results")
    print("Hybrid search PASSED")
else:
    print("\nLexical/Hybrid: SKIPPED (server doesn't support sparse indexing)")

=== Vector Search ===
Found 3 results for 'how to set up feature flags'
  1. (67 chars) There are 3 steps to implement feature flags using the PostHog API:...
  2. (1388 chars) There are 2 steps to implement feature flags in PHP:

### Step 1: Evaluate the feature flag value  
#### Boolean feature...
  3. (1297 chars) ---
title: Feature flag dependencies
sidebar: Docs
showTitle: true
availability:
free: full
selfServe: full
enterprise: ...
Vector search PASSED

=== search_text() ===
Found 3 results
search_text PASSED

=== search_content() ===
Got 3 content strings
search_content PASSED

Lexical/Hybrid: SKIPPED (server doesn't support sparse indexing)


In [10]:
# --- File-structure awareness ---
print("=== File-Structure Awareness ===")
file_aware = source._files.check()
print(f"File-aware: {file_aware}")

if file_aware:
    all_paths = source._files.get_all_file_paths()
    print(f"Found {len(all_paths)} unique file paths")
    for p in all_paths[:5]:
        print(f"  {p}")
    if len(all_paths) > 5:
        print(f"  ... and {len(all_paths) - 5} more")

    top_level = source.get_top_level_chunks()
    print(f"\nTop-level chunks: {len(top_level)}")

    # Chunk context
    ctx = source.get_chunk_with_context(samples[0])
    print(f"\n=== Chunk Context ===")
    print(f"  prev: {ctx['prev_chunk_preview'][:80]}...")
    print(f"  next: {ctx['next_chunk_preview'][:80]}...")
    print("Context check PASSED")
else:
    print("Skipping file-structure tests")

# --- search_related ---
print("\n=== search_related ===")
related = source.search_related(samples[0], ["feature flags", "analytics"], top_k=3)
print(f"Found {len(related)} related chunks")
for r in related:
    print(f"  queries={r['queries']}, same_file={r['same_file']}, score={r['max_score']:.3f}")
print("search_related PASSED")

# --- Filtered search ---
if file_aware and all_paths:
    print("\n=== Filtered Search ===")
    target_path = all_paths[0]
    filt_results = source.search_text(
        "documentation", top_k=5,
        filter=FieldPredicate(field="file_path", op="eq", value=target_path),
    )
    print(f"Filtered to '{target_path}': {len(filt_results)} results")
    for chunk in filt_results:
        assert chunk.get_metadata("file_path") == target_path
    print("Filtered search PASSED")

print("\n=== All Step 1 checks PASSED ===")

=== File-Structure Awareness ===
File-aware: True
Found 1264 unique file paths
  _snippets/capture-group-event-code.mdx
  _snippets/exposed-api-keys.mdx
  _snippets/groups-intro.mdx
  _snippets/groups-use-cases.mdx
  _snippets/groups-vs-cohorts.mdx
  ... and 1259 more

Top-level chunks: 39

=== Chunk Context ===
  prev: {
  "char_count": 1024,
  "chunk_hash": "b3924f8fa97f2a5522ea27772eed4640f29b88f...
  next: (No next chunk)...
Context check PASSED

=== search_related ===
Found 6 related chunks
  queries=['feature flags'], same_file=False, score=0.768
  queries=['feature flags'], same_file=False, score=0.757
  queries=['feature flags'], same_file=False, score=0.753
  queries=['analytics'], same_file=False, score=0.719
  queries=['analytics'], same_file=False, score=0.712
  queries=['analytics'], same_file=False, score=0.708
search_related PASSED

=== Filtered Search ===
Filtered to '_snippets/capture-group-event-code.mdx': 4 results
Filtered search PASSED

=== All Step 1 checks PASSED

## Step 2: Create Synthetic QA

Generate synthetic QA pairs with `CgftPipeline` using the Chroma corpus from Step 1.

In [11]:
from cgft.qa_generation.cgft_models import (
    CgftPipelineConfig,
    CorpusConfig,
    CorpusContextConfig,
    FilteringConfig,
    GenerationConfig,
    GroundingLLMFilterConfig,
    LLMDirectGenerationConfig,
    OutputConfig,
    PlatformConfig,
    RetrievalLLMFilterConfig,
    TargetsConfig,
)
from cgft.qa_generation.cgft_pipeline import CgftPipeline

cfg = CgftPipelineConfig(
    platform=PlatformConfig(
        api_key=API_KEY, base_url=BASE_URL,
        llm_api_key=LLM_API_KEY, llm_base_url=LLM_BASE_URL,
    ),
    corpus=CorpusConfig(corpus_name="test", min_chunk_chars=400),
    corpus_context=CorpusContextConfig(
        description=CORPUS_DESCRIPTION,
        example_queries=EXAMPLE_QUERIES,
        num_top_level_samples=4,
        num_random_samples=4,
        min_chunk_chars=400,
    ),
    generation=GenerationConfig(
        llm_direct=LLMDirectGenerationConfig(model=LLM_MODEL),
    ),
    filtering=FilteringConfig(
        retrieval_llm=RetrievalLLMFilterConfig(judge_model=LLM_MODEL),
        grounding_llm=GroundingLLMFilterConfig(judge_model=LLM_MODEL),
    ),
    targets=TargetsConfig(
        total_samples=TOTAL_SAMPLES,
        qa_type_distribution={
            "lookup": 1,
            "co_located_multi_hop": 0,
            "cross_document_multi_hop": 0,
            "sequential_reasoning": 0,
            "synthesis": 0.0,
        },
    ),
    output=OutputConfig(dir=OUTPUT_DIR, train_jsonl="train.jsonl", eval_jsonl="eval.jsonl"),
)

cfg.resolve_api_keys()
print("generator model:", cfg.generation.llm_direct.model)
print("generator base_url:", cfg.generation.llm_direct.base_url)
print("retrieval judge:", cfg.filtering.retrieval_llm.judge_model)
print("grounding judge:", cfg.filtering.grounding_llm.judge_model)

generator model: grok-4-fast-reasoning
generator base_url: https://judge-elsa.services.ai.azure.com/openai/v1/
retrieval judge: grok-4-fast-reasoning
grounding judge: grok-4-fast-reasoning


In [12]:
import importlib

import cgft.qa_generation.cgft_pipeline as _cgft_pipeline_mod
import cgft.qa_generation.generators.direct_llm as _direct_llm_mod

importlib.reload(_direct_llm_mod)
importlib.reload(_cgft_pipeline_mod)
CgftPipeline = _cgft_pipeline_mod.CgftPipeline

cgft_pipeline = CgftPipeline(cfg, source_factory=lambda _cfg: source)
result = cgft_pipeline.run()

train_data = result["train_dataset"]
eval_data = result["eval_dataset"]
stats = result["stats"]

[1/8] Loading chunks from corpus...
[1/8] Loaded 30 seed chunks from corpus
[2/8] Building corpus profile...
[2/8] Built corpus profile
[3/8] Generating entity patterns...
[3/8] Generated entity patterns: 9 entities, 38 patterns
[4/8] Created 30 generation tasks
[5/8] Generating 30 QA candidates...


Linking chunks:   0%|          | 0/30 [00:00<?, ?it/s]

Processing prompts:   0%|          | 0/30 [00:00<?, ?it/s]

Skipping task task_00019: failed to parse QA response.
Skipping task task_00010: failed to parse QA response.
Skipping task task_00026: failed to parse QA response.
Skipping task task_00022: failed to parse QA response.
Skipping task task_00005: failed to parse QA response.
Skipping task task_00012: failed to parse QA response.
Skipping task task_00015: failed to parse QA response.
Skipping task task_00025: failed to parse QA response.
Skipping task task_00029: failed to parse QA response.
Skipping task task_00021: failed to parse QA response.
Skipping task task_00027: failed to parse QA response.
Skipping task task_00001: failed to parse QA response.
Skipping task task_00013: failed to parse QA response.
Skipping task task_00018: failed to parse QA response.
Skipping task task_00017: failed to parse QA response.
Skipping task task_00028: failed to parse QA response.
Skipping task task_00004: failed to parse QA response.
Skipping task task_00024: failed to parse QA response.
Skipping t

[5/8] Generated 9 QA candidates (21 failed to parse)
[6/8] Starting filtering (max 4 rounds)...
  Round 1/5: 9 items to evaluate
    deterministic_guards: 9 passed, 0 need refinement, 0 rejected
    grounding_llm: evaluating 0 items...


Processing prompts:   0%|          | 0/9 [00:00<?, ?it/s]

    grounding_llm: 9 passed, 0 need refinement, 0 rejected
    retrieval_too_easy_llm: evaluating 0 items...


Processing prompts:   0%|          | 0/9 [00:00<?, ?it/s]

    retrieval_too_easy_llm: 9 passed, 0 need refinement, 0 rejected
[7/8] Transformed 9 passed items
[8/8] Formatted output: 7 train, 2 eval

Pipeline complete: 9 passed, 0 rejected (0 regenerations)


### Step 2a: Verify Search Environment

Test the RL environment independently before launching training.

Uses `SearchEnv` with `ChromaSearch` — the new unified pattern.
The legacy `ChromaSearchEnv` also works (it's a thin wrapper).

In [ ]:
import asyncio
import nest_asyncio
nest_asyncio.apply()

from cgft.corpus.chroma.search import ChromaSearch
from cgft.envs.search_env import SearchEnv

# Option 1 (recommended): SearchEnv + ChromaSearch
search = ChromaSearch(
    collection_name=COLLECTION_NAME,
    host=CHROMA_HOST,
    port=CHROMA_PORT,
    embed_fn=EMBED_FN,
    enable_bm25=True,
)
env = SearchEnv(
    search=search,
    judge_base_url=JUDGE_BASE_URL,
    judge_api_key=JUDGE_API_KEY,
    judge_model=JUDGE_MODEL,
)

# Option 2 (legacy, still works):
# from cgft.envs.chroma_search_env import ChromaSearchEnv
# env = ChromaSearchEnv(
#     collection_name=COLLECTION_NAME, host=CHROMA_HOST,
#     port=CHROMA_PORT, embed_fn=EMBED_FN,
#     judge_base_url=JUDGE_BASE_URL, judge_api_key=JUDGE_API_KEY,
#     judge_model=JUDGE_MODEL,
# )

# --- List tools ---
tools = asyncio.run(env.list_tools())
print("=== Search Environment Tools ===")
for tool in tools:
    print(f"  {tool.name}: {tool.description}")
    print(f"    schema: {tool.input_schema}")
print(f"Default mode: {env._default_mode}")
print("Tool listing PASSED")

# --- Run search tool ---
print("\n=== Search Tool Execution ===")
search_result = asyncio.run(env._search_tool(query="how to install posthog"))
print(search_result[:500])
assert not search_result.startswith("Error")
print("\nSearch tool PASSED")

# --- Empty query ---
print("\n=== Empty Query ===")
empty_result = asyncio.run(env._search_tool(query=""))
assert empty_result.startswith("Error")
print(f"Correctly rejected: {empty_result}")
print("Empty query PASSED")

# --- Dataset preprocessing ---
print("\n=== Dataset Preprocessing ===")
std = SearchEnv.dataset_preprocess({"question": "What is X?", "answer": "Y"})
assert std["prompt"] == "What is X?"
assert std["ground_truth"] == "Y"
print("Preprocessing PASSED")

print("\n=== All environment checks PASSED ===")

## Step 3: Launch Training

Upload datasets and environment, then launch the training job.

> **Important**: The Chroma server must be accessible from the training cluster. If you're running Chroma locally (`localhost`), remote validation will fail because the CGFT cloud can't reach your machine. Options:
> 1. Use a cloud-hosted Chroma instance (Chroma Cloud or a server with a public URL)
> 2. Skip remote validation with `validate_env_remotely=False` (training will still fail if the server is unreachable from the cluster)
> 3. For local testing, use `validate_env_remotely=False` and `validate_env=True` to test locally only

In [ ]:
import cgft
from cgft.corpus.chroma.search import ChromaSearch
from cgft.envs.search_env import SearchEnv
from cgft.trainer.pipeline import train

# Build the search client (pickle-safe — survives remote training)
search = ChromaSearch(
    collection_name=COLLECTION_NAME,
    host=CHROMA_HOST,
    port=CHROMA_PORT,
    embed_fn=EMBED_FN,
    enable_bm25=True,
)

# Set to False if your Chroma server is on localhost or behind a firewall
# that CGFT's cloud cannot reach. For Chroma Cloud or a public server, use True.
VALIDATE_REMOTELY = True

experiment_id = train(
    env_class=SearchEnv,
    env_args={
        "search": search,
        "judge_base_url": JUDGE_BASE_URL,
        "judge_api_key": JUDGE_API_KEY,
        "judge_model": JUDGE_MODEL,
    },
    train_dataset=train_data,
    eval_dataset=eval_data,
    prefix=EXPERIMENT_PREFIX,
    api_key=API_KEY,
    base_url=BASE_URL,
    local_modules=[cgft],
    pip_dependencies=["chromadb", "openai"],
    experiment_name=EXPERIMENT_NAME,
    validate_env_remotely=VALIDATE_REMOTELY,
)

# Legacy alternative (also works):
# from cgft.envs.chroma_search_env import ChromaSearchEnv
# train(env_class=ChromaSearchEnv, env_args={...}, ...)

## Monitoring Training: What to Expect

### Early Training Noise

**Early training**: At the start of a training run, rewards will fluctuate and metrics may be noisy. This is completely normal - the model is still learning basic patterns and its outputs are unstable. Give it some time (usually a few dozen steps) and the signal will clean up and you'll start seeing clear trends.

**Exploration before exploitation**: Ideally, you want to see pass@k climbing first before average reward increases significantly. This means your model is exploring the solution space and learning to solve more prompts (breadth) before it optimizes for higher rewards on those prompts (depth). If average reward shoots up while pass@k stays low, you're likely exploiting a narrow set of easy prompts rather than building robust capabilities.

**Healthy training trajectory**: In a well-configured training run, expect pass@k to increase early as the model learns to solve more distinct prompts. Average reward should follow but lag behind. Eventually both should plateau as the model saturates your training distribution.

**Warning signs**:

- pass@k flat while average reward increases → model is overfitting to a narrow subset of prompts
- both metrics stagnate early → training distribution may be too hard, reward signal too sparse